# Topic declaration

**Question:** Which of the seven parameters significantly change peak width or height, and are they the same for early- and late-eluting standards?

**Data:** Gamble, S.; Granger, C.; Mannion, J. *GC-MS Screening Data for Instrument Optimization*. figshare, 2025. https://doi.org/10.6084/m9.figshare.28635920.v1 (license GPL 3.0+). Accessed 2026-11-08. Associated article: *Anal. Chem.*, https://doi.org/10.1021/acs.analchem.3c05763.

The dataset holds 32 GC-MS total ion chromatograms of 11 iodinated standards, one per method of a two-level fractional factorial design in seven instrument parameters (`ffd.csv`), and the retention time of each standard in each method (`rts.csv`).

## Load the data

Purpose: load the design table, the retention-time table and one chromatogram from `data/`, exactly as downloaded, and report their size, columns and units.

Scientific choice: the chromatogram files begin with two text lines (the method name and "Start of data points") and a byte-order mark, so they are read with `skiprows=2` and `encoding="utf-8-sig"`; nothing in the files is edited. Method 1 is shown as an example; all 32 share this format.

Expected output: 32 rows and columns `Method`, `A`–`G` for the design; 32 rows and 11 standards for the retention times; about 16,000 time points for Method 1, with time in minutes and intensity in counts.

In [1]:
import pandas as pd

design = pd.read_csv("data/ffd.csv")  # factor levels per method, columns A-G
retention_times = pd.read_csv("data/rts.csv", index_col="Chemical")  # min
tic = pd.read_csv(
    "data/TIC_Method_1.csv",
    skiprows=2,
    header=None,
    names=["time_min", "intensity_counts"],
    encoding="utf-8-sig",
)  # time in min, total ion current in counts

print(f"Design (ffd.csv): {len(design)} rows")
print(list(design.columns))
print()
print(f"Retention times (rts.csv): {len(retention_times)} rows")
print(list(retention_times.columns))
print()
print(f"Chromatogram (TIC_Method_1.csv): {len(tic)} rows")
print(list(tic.columns))
print(f"time range: {tic.time_min.min():.2f} to {tic.time_min.max():.2f} min")
print("time: min; intensity: counts (total ion current); retention time: min")

Design (ffd.csv): 32 rows
['Method', 'A', 'B', 'C', 'D', 'E', 'F', 'G']

Retention times (rts.csv): 32 rows
['iodopropane', 'iodobutane', 'iodopentane', 'iodohexane', 'iodobenzene', 'iodoheptane', 'iodooctane', 'iodononane', 'iododecane', 'iodoundecane', 'iodododecane']

Chromatogram (TIC_Method_1.csv): 16245 rows
['time_min', 'intensity_counts']
time range: 4.00 to 63.00 min
time: min; intensity: counts (total ion current); retention time: min


## Is the sampling fine enough to measure peak width?

Purpose: check whether the chromatograms have enough points across each peak to measure its width, for the sufficiency section of the proposal.

Scientific choice: for each of the 11 standards in each of the 32 methods, the peak is taken within `PEAK_WINDOW_MIN` of its retention time in `rts.csv`, the baseline is the lowest intensity in that window, and the points at or above half of the apex height above that baseline are counted. This is a rough screen, not the final peak measurement; a co-eluting peak inside the window would inflate the count.

Expected output: the time step in minutes, the number of points in a short (Method 2) and a long (Method 1) run, and the minimum, median and maximum number of points across half height over all 352 peaks.

In [2]:
# Parameters
N_METHODS = 32  # number of runs in the design (ffd.csv)
PEAK_WINDOW_MIN = 0.2  # min; half-width of the search window around each retention time, wider than any peak seen (~0.07 min at half height)
HALF_HEIGHT_FRACTION = 0.5  # width is counted at half the apex height above baseline (FWHM convention)


def read_tic(method):
    """Read one chromatogram as downloaded: time in min, intensity in counts."""
    return pd.read_csv(
        f"data/TIC_Method_{method}.csv",
        skiprows=2,
        header=None,
        names=["time_min", "intensity_counts"],
        encoding="utf-8-sig",
    )


points_across_peak = []  # number of points at or above half height, one per peak
for method in range(1, N_METHODS + 1):
    chromatogram = read_tic(method)
    for standard in retention_times.columns:
        rt_min = retention_times.loc[f"Method {method}", standard]  # min
        window = chromatogram[(chromatogram.time_min - rt_min).abs() <= PEAK_WINDOW_MIN]
        baseline = window.intensity_counts.min()  # counts
        apex = window.intensity_counts.max()  # counts
        half_height = baseline + HALF_HEIGHT_FRACTION * (apex - baseline)  # counts
        points_across_peak.append(int((window.intensity_counts >= half_height).sum()))

points_across_peak = pd.Series(points_across_peak)
time_step_min = tic.time_min.diff().median()  # min

print(f"time step: {time_step_min:.5f} min ({time_step_min * 60:.2f} s)")
print(f"points in Method 2 (short run): {len(read_tic(2))}")
print(f"points in Method 1 (long run): {len(tic)}")
print(f"peaks checked: {len(points_across_peak)}")
print(
    "points across half height: "
    f"min {points_across_peak.min()}, "
    f"median {points_across_peak.median():.0f}, "
    f"max {points_across_peak.max()}"
)

time step: 0.00363 min (0.22 s)
points in Method 2 (short run): 10325
points in Method 1 (long run): 16245
peaks checked: 352
points across half height: min 6, median 11, max 27


## Environment

Purpose: record the Python and package versions this notebook ran with. Expected output: Python 3.14 and the pandas version from `uv.lock`.

In [3]:
import sys

print(f"Python {sys.version.split()[0]}")
print(f"pandas {pd.__version__}")

Python 3.14.7
pandas 3.0.6
